<a href="https://colab.research.google.com/github/dorionrichard33-glitch/CIS3120/blob/main/Module10_ForeignKeys_Normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 10 — Foreign Keys, Normalization, and Data Integrity

**CIS 3120 · Programming for Analytics · Baruch College / Zicklin School of Business**


<i>a copy of this notebook is available at [bit.ly/cis3120_module10](https://bit.ly/cis3120_module10)</i>

---

This notebook extends the database skills you developed in Module 7.  You will work
with multi-table schemas, enforce referential integrity using foreign keys, recognize
and repair normalization problems, and write queries that traverse relationships across
tables.

**How to use this notebook**

- Read each instruction cell before running or writing any code.
- Every exercise cell contains a `# TODO` comment that marks exactly where your work goes.
- Run cells in order from top to bottom. Later exercises depend on tables created earlier.
- If you make a mistake and need to start a section over, re-run the setup cell at the
  top of that section to recreate the database.

**Sections**

1. Environment Setup  
2. Foreign Key Mechanics  
3. Normalization — Recognition and Repair  
4. Multi-Table Querying with Real Relationships  
5. Data Integrity Scenarios  
6. Schema Design Challenge  


---
## Section 1 — Environment Setup and RAM-only Databases

For housekeeping purposes, we can rely on a function definitions to prepare the environment for each connection and another functoin we define to show all tables in the active connection.  That second function may be helpful for us when working with RAM-only `:memory:` databases that we would not be able to view in a standalone tool such as DBbrowser which works with databases on secondary storage, not in RAM.  
- `run(conn, sql)` — executes a SQL statement and prints the results.  
- `show_tables(conn)` — lists all tables currently in the database.

Regarding RAM-only databases, we will take a closer look at their use, including some ways that we can replicate their contents into databases that are stored on disk.  

<div style="background:#fff8e1;border-left:5px solid #f9a825;padding:10px 16px;margin:6px 0;border-radius:4px;">

**════════════════════════════════════════════════════════════════**  
**INSTRUCTOR NOTES**  
**════════════════════════════════════════════════════════════════**

**Purpose of this section**

This section exists solely to establish shared infrastructure.  There are no student
exercises here.  Draw students' attention to two points:

1. **`sqlite3` is part of the Python standard library** — no `pip install` required.  
   This is a deliberate design choice: students can run the notebook anywhere Python
   is available, including Colab, without additional setup.

2. **In-memory databases** (`":memory:"`) are used throughout so that each section
   starts from a known state.  Emphasise that `:memory:` databases are destroyed when
   the Python process ends — they are appropriate for exercises but not for persistent
   storage.

**Common student question**: "Why not just use pandas?"  
Brief answer: pandas is excellent for analysis, but it does not enforce schema
constraints, foreign keys, or referential integrity.  SQL databases are the appropriate
tool when data integrity rules matter.

</div>  

---

### 1.1 Housekeeping -- Environment Setup  
<i>Functions For Use Throughout This Notebook</i>>  


Run the cell below to import `sqlite3` and define two helper functions you will
use throughout the notebook:

- `run(conn, sql)` — executes a SQL statement and prints any results as a formatted table.
- `show_tables(conn)` — lists all tables currently in the database.

You do not need to modify this cell.


In [8]:
# Housekeeping: a single execution to import Python's standard `sqlite3` module
import sqlite3

# Housekeeping: define a `run()` function to execute a SQL statement and print
def run(conn, sql, params=()):
    """Execute a SQL statement and print results, if any.
    Prevents SQL injection and allows dynamic values to be passed in.
    Arguments: conn   - Represents the active database session,
               sql    - The SQL statement to execute, and
               params - Parameterized query inputs,"""
    try:
        cur = conn.cursor()       # Assign db connection cursor to a local variable
        cur.execute(sql, params)  # Invoke execute() to bind params to placeholders (?) in the SQL string
        conn.commit()             # Invoke commit() to commits the transaction - moot for SELECT
        rows = cur.fetchall()     # Invoke fetchall() to retrieve all result rows as tuples
        if rows:                  # If there are rows of results, print them
            cols = [d[0] for d in cur.description]     # Metadata about columns returned by the query
            # The following lines tinker with the formatting of output to print
            col_widths = [max(len(str(c)), max((len(str(r[i])) for r in rows), default=0))
                          for i, c in enumerate(cols)]
            header = "  ".join(str(c).ljust(col_widths[i]) for i, c in enumerate(cols))
            separator = "  ".join("-" * col_widths[i] for i in range(len(cols)))
            print(header)
            print(separator)
            # The following loop iterates through rows of results and prints using the tinkering above
            for row in rows:
                print("  ".join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
            print(f"\n({len(rows)} row{'s' if len(rows) != 1 else ''})")
        else:                   # If there are no rows of results returned, not much to do
            print("Statement executed successfully. No rows returned.")
    except sqlite3.Error as e:
        print(f"Database error occurred: {e}")
        # Rollback the transaction in case of a database error
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        # Rollback in case of any other unexpected error
        if conn:
            conn.rollback()

def show_tables(conn):
    """List all tables in the database."""
    try:
        # Using the run function defined above, SELECT data from `sqlite_master` table
        # Query returns records where the value of the `type` field is `table` and sorts
        run(conn, "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    except Exception as e:
        print(f"Error displaying tables: {e}")

def close_connection(conn):
    """Closes a SQLite connection, implicitly committing any pending transactions.
    Arguments: conn - The database connection object to close.
    """
    if conn:
        try:
            conn.close()
            print("Connection closed.")
        except Exception as e:
            print(f"Error closing connection: {e}")
    else:
        print("No connection to close.")


### 1.2 RAM Only Databases -- `:memory:`  

####About `:memory:` RAM Only Databases  
  
Using `:memory:` as the SQLite database name creates a database that lives entirely in RAM, not on disk. It is deleted when the connection closes, and each connection normally gets its own separate in-memory database.  
  
A basic example is:  
  
```  
import sqlite3  
conn = sqlite3.connect(":memory:")  
```  
  
In SQLite documentation, `:memory:` must be used exactly, with no extra path text, or it will not behave as a pure in-memory database.

For SQL statements inside the active connection, the primary database is referred to as `main`, even when that database was created with `:memory:`.

So you would use things like:

```
CREATE TABLE main.users(id INTEGER PRIMARY KEY, name TEXT);  
INSERT INTO main.users(name) VALUES ('Alice');  
SELECT * FROM main.users;  
```

You can usually omit main. because it is the default database for unqualified table names in that connection.  

`temp` is the separate temporary database name, while `main` is the attached primary database name for the connection.  More about `temp` later -- it can be a useful tool for processing efficiency and reliability.   

One important distinction: `:memory:` is the filename used to create the database, but `main` is the SQL schema name you use to refer to it during the connection.  

####1.2.a Creating/Using RAM-only Databases  

In the following code:  
1. We open a connection to a SQLite database in RAM using ":memory:" as the argument for the `.connect()` method.  
2. Then we create a table named `users` in that RAM database referred to as `main.` with dot-notation and create an `id` integer field and a `name` text field on the table.  
3. We insert the value 'Alice' into the `name` field on the table `users` in that RAM database referred to as `main.` with dot-notation.  
4. Finally, we select all records from the `users` table in that RAM database referred to as `main.` with dot-notation.  

In [14]:
conn = sqlite3.connect(":memory:")
run(conn, "CREATE TABLE main.users(id INTEGER PRIMARY KEY, name TEXT);")
run(conn, "INSERT INTO main.users(name) VALUES ('Alice');")
run(conn, "SELECT * FROM main.users;")

Statement executed successfully. No rows returned.
Statement executed successfully. No rows returned.
id  name 
--  -----
1   Alice

(1 row)


In [15]:
show_tables(conn)

name 
-----
users

(1 row)


In [16]:
close_connection(conn)

Connection closed.


In [17]:
show_tables(conn)

Database error occurred: Cannot operate on a closed database.
Error displaying tables: Cannot operate on a closed database.


Alternatively, we could have opened the RAM-only database and worked with it in the active connection without the need for dot-notation and the reference to `main.`  

In [19]:
conn = sqlite3.connect(":memory:")
run(conn, "CREATE TABLE users(id INTEGER PRIMARY KEY, name TEXT);")
run(conn, "INSERT INTO users(name) VALUES ('Alice');")
run(conn, "SELECT * FROM users;")

Statement executed successfully. No rows returned.
Statement executed successfully. No rows returned.
id  name 
--  -----
1   Alice

(1 row)


In [20]:
type(conn)

sqlite3.Connection



---



####1.2.b Replicating RAM-only Databases to Disk  

There are several mechanisms for replicating the content of a RAM-only database to secondary storage:  
1. VACUUM statement  
2. Built-in `backup()` method  

First, let's try a simple example using the `VACUUM` statement in SQLite. Look at your current working directory before running the following code -- there should not be a file named `saved.db` in your current working directory. If there is, you could be at risk of losing the data in that file and should change the argument in the `VACUUM` statement below.

 Bear in mind that the VACUUM statement can also be applied to databases that are on secondary storage (disk) and will make a backup copy of any open database.    

In [25]:
run(conn, "VACUUM main INTO 'saved.db'")

Statement executed successfully. No rows returned.


After running the code above, you should see a file named `saved.db` in your current working directory.  Let's open a connection to that database file and use our `show_tables()` function to see what tables it includes.  


In [27]:
connSaved = sqlite3.connect("saved.db")
show_tables(connSaved)

name 
-----
users

(1 row)


We can use `SELECT` statements to query the `users` tables in the two database connections we have open.

In [29]:
print("=== RAM-only database ===")
run(conn, "SELECT * FROM users;")
print("\n=== Saved database ===")
run(connSaved, "SELECT * FROM users;")

=== RAM-only database ===
id  name 
--  -----
1   Alice

(1 row)

=== Saved database ===
id  name 
--  -----
1   Alice

(1 row)


Before going further, let's close the connection to that 'saved.db' database and delete the file. We will keep our RAM-only `main` database connection open because we will use it again to demonstrate the used of the `.backup()` method.  

In [31]:
close_connection(connSaved)
import os                      # importing the os - operating system module
try:
    os.remove("saved.db")      # delete file using the os.remove() function
    print("saved.db deleted.")
except FileNotFoundError:
    print("saved.db was not found, no file to delete.")
except Exception as e:
    print(f"An error occurred while deleting saved.db: {e}")

print("Current working directory:")
os.listdir()                   # list the contents of the current working directory


Connection closed.
saved.db deleted.
Current working directory:


['.config',
 'libraryhomework.ipynb',
 'Music',
 'CIS3120',
 '.condarc',
 'mynewdirectory',
 'Loan.csv',
 'MyFirstNotebook.ipynb',
 '.DS_Store',
 'complaints.ipynb',
 '.CFUserTextEncoding',
 'pets.db',
 '.xonshrc',
 'anaconda_projects',
 '.zshrc',
 'CodingGraceGame',
 'Pandas Data Analysis.ipynb',
 'my_311_data.csv',
 'CIS2300',
 'Pictures',
 '.zprofile',
 '.zsh_history',
 '.ipython',
 'Desktop',
 'Library',
 'library.db',
 'Module10_ForeignKeys_Normalization.ipynb',
 'Public',
 'Pandas_ICE.ipynb',
 '.idlerc',
 '.tcshrc',
 '.anaconda',
 '.ssh',
 'Movies',
 'Applications',
 'Member.csv',
 '.Trash',
 'NYCOpenData_Apr0625.ipynb',
 '.ipynb_checkpoints',
 '.jupyter',
 'Book.csv',
 'potatohead.png',
 'Documents',
 'homework.ipynb',
 '.bash_profile',
 'Downloads',
 '.python_history',
 '.continuum',
 '.gitconfig',
 '.zsh_sessions',
 '.conda',
 '5uac-w243.csv']

Let's use the built-in `.backup()` methd to write a copy of the `main` database to disk.  Bear in mind that this method can also be applied to databases that are on secondary storage (disk) and will make a backup copy of any open database.  

In [33]:
target_conn = sqlite3.connect("saved.db")
conn.backup(target_conn)
print("=== RAM-only database ===")
show_tables(conn)
print("\n=== Saved database ===")
show_tables(target_conn)

=== RAM-only database ===
name 
-----
users

(1 row)

=== Saved database ===
name 
-----
users

(1 row)


In [34]:
close_connection(target_conn)
import os                      # importing the os - operating system module
try:
    os.remove("saved.db")      # delete file using the os.remove() function
    print("saved.db deleted.")
except FileNotFoundError:
    print("saved.db was not found, no file to delete.")
except Exception as e:
    print(f"An error occurred while deleting saved.db: {e}")

print("Current working directory:")
os.listdir()                   # list the contents of the current working directory


Connection closed.
saved.db deleted.
Current working directory:


['.config',
 'libraryhomework.ipynb',
 'Music',
 'CIS3120',
 '.condarc',
 'mynewdirectory',
 'Loan.csv',
 'MyFirstNotebook.ipynb',
 '.DS_Store',
 'complaints.ipynb',
 '.CFUserTextEncoding',
 'pets.db',
 '.xonshrc',
 'anaconda_projects',
 '.zshrc',
 'CodingGraceGame',
 'Pandas Data Analysis.ipynb',
 'my_311_data.csv',
 'CIS2300',
 'Pictures',
 '.zprofile',
 '.zsh_history',
 '.ipython',
 'Desktop',
 'Library',
 'library.db',
 'Module10_ForeignKeys_Normalization.ipynb',
 'Public',
 'Pandas_ICE.ipynb',
 '.idlerc',
 '.tcshrc',
 '.anaconda',
 '.ssh',
 'Movies',
 'Applications',
 'Member.csv',
 '.Trash',
 'NYCOpenData_Apr0625.ipynb',
 '.ipynb_checkpoints',
 '.jupyter',
 'Book.csv',
 'potatohead.png',
 'Documents',
 'homework.ipynb',
 '.bash_profile',
 'Downloads',
 '.python_history',
 '.continuum',
 '.gitconfig',
 '.zsh_sessions',
 '.conda',
 '5uac-w243.csv']

<b>Summary: `VACUUM` vs. `.backup()`</b>   
A concise distinction centers on purpose (optimization vs. preservation) and mechanism (rebuild vs. copy).  
  
---

#####`VACUUM` (SQL statement)

Primary use: reclaim space and optimize the database file

* Rebuilds the entire database file into a new, compact form
* Eliminates free pages left by deletions or updates
* Defragments data → can improve I/O performance
* Can change page size or encoding (when specified)
* Operates *within* SQLite as a SQL command

When to choose it:

* After large deletes or updates
* To reduce file size
* To improve query performance through defragmentation
  
---
#####`.backup` (SQLite shell command or API)

Primary use: create a consistent copy of the database

* Copies the database to another file (logical snapshot)
* Preserves current structure and content without reorganization
* Can run while the database is in use (non-blocking in many cases)
* Does not reclaim space or optimize layout

When to choose it:

* To create backups or snapshots
* For migration or duplication
* For recovery or versioning purposes

---
##### Core Distinction

* `VACUUM` → internal optimization and compaction
* `.backup` → external duplication and preservation

They address different concerns and are often complementary rather than interchangeable.


---
## Section 2 — Foreign Key Mechanics

### Background

A **foreign key** is a column (or set of columns) in one table whose values must match
values in the primary key of another table.  This relationship is called
**referential integrity**: the database will reject any operation that would leave a
child row pointing to a parent row that does not exist.

SQLite does not enforce foreign keys by default.  You must enable enforcement at the
start of each connection with:

```sql
PRAGMA foreign_keys = ON;
```

In this section you will:

1. Create a two-table schema with a foreign key constraint.  
2. Observe what happens when you try to insert a row that violates the constraint.  
3. Correct the insert and confirm the relationship works as intended.

### The scenario

A small veterinary clinic tracks **owners** and their **pets**.  Every pet record must
reference a valid owner.

---



### Exercise 2.1 — Enable foreign key enforcement and create the schema

In [38]:
conn2 = sqlite3.connect(":memory:")
conn2.execute("PRAGMA foreign_keys = ON;")
print("Foreign key enforcement is ON.")


Foreign key enforcement is ON.


The two tables you need are defined below.  Study the schema carefully, then run the
cell to create them.

```sql
CREATE TABLE Owner (
    owner_id   INTEGER PRIMARY KEY,
    full_name  TEXT    NOT NULL,
    phone      TEXT
);

CREATE TABLE Pet (
    pet_id     INTEGER PRIMARY KEY,
    pet_name   TEXT    NOT NULL,
    species    TEXT    NOT NULL,
    owner_id   INTEGER NOT NULL,
    FOREIGN KEY (owner_id) REFERENCES Owner(owner_id)
);
```

Notice that `Pet.owner_id` is declared as a `FOREIGN KEY` that references `Owner.owner_id`.
Any `pet_id` inserted into `Pet` must correspond to an existing row in `Owner`.


In [40]:
conn2.execute("DROP TABLE IF EXISTS Pet;")
conn2.execute("DROP TABLE IF EXISTS Owner;")

conn2.execute("""
    CREATE TABLE Owner (
        owner_id   INTEGER PRIMARY KEY,
        full_name  TEXT    NOT NULL,
        phone      TEXT
    );
""")

conn2.execute("""
    CREATE TABLE Pet (
        pet_id     INTEGER PRIMARY KEY,
        pet_name   TEXT    NOT NULL,
        species    TEXT    NOT NULL,
        owner_id   INTEGER NOT NULL,
        FOREIGN KEY (owner_id) REFERENCES Owner(owner_id)
    );
""")

conn2.commit()
show_tables(conn2)


name 
-----
Owner
Pet  

(2 rows)


Let's push a copy of the `main` database in RAM to a database file 'pets.db'

In [42]:
run(conn2, "VACUUM main INTO 'pets.db'")
show_tables(conn2)

Database error occurred: output file already exists
name 
-----
Owner
Pet  

(2 rows)


After downloading 'pets.db' to your computer, you can view its contents with DBbrowser or any SQLite utility.

---

### Exercise 2.2 — rowid, AUTOINCREMENT, and SQLite system tables

SQLite assigns every row an internal integer identifier called **rowid**.  When you
declare a column as `INTEGER PRIMARY KEY`, that column becomes an alias for `rowid`
and SQLite fills it in automatically if you do not supply a value.

Adding the `AUTOINCREMENT` keyword strengthens this guarantee: SQLite records the
highest ID ever used in a system table called `sqlite_sequence` and never reuses
that value, even after a row is deleted.

A second system table, `sqlite_master`, is SQLite's internal catalog of every table,
index, view, and trigger in the database.

In this exercise you will:

1. Create a table **with** `AUTOINCREMENT` and observe `sqlite_sequence`.
2. Insert rows without supplying an ID and confirm SQLite fills it in.
3. Delete a row, then observe that the next inserted ID does **not** reuse the
   deleted value.
4. Query `sqlite_master` to inspect the schema of your own tables.


In [45]:
# Setup — fresh connection for this exercise
conn2b = sqlite3.connect(":memory:")
conn2b.execute("PRAGMA foreign_keys = ON;")

# Part A: create a table using AUTOINCREMENT
conn2b.execute("""
    CREATE TABLE Animal (
        animal_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        common_name TEXT    NOT NULL,
        species     TEXT    NOT NULL
    );
""")
conn2b.commit()

# Part B: insert three rows WITHOUT supplying animal_id
conn2b.executemany(
    "INSERT INTO Animal (common_name, species) VALUES (?, ?);",
    [
        ("Mango",   "Canis lupus familiaris"),
        ("Shadow",  "Felis catus"),
        ("Biscuit", "Oryctolagus cuniculus"),
    ]
)
conn2b.commit()

print("=== Animal table after three inserts ===")
run(conn2b, "SELECT * FROM Animal;")

print("\n=== sqlite_sequence ===")
run(conn2b, "SELECT * FROM sqlite_sequence;")

# Part C: delete row with animal_id = 3, then insert a new row
conn2b.execute("DELETE FROM Animal WHERE animal_id = 3;")
conn2b.execute("INSERT INTO Animal (common_name, species) VALUES ('Noodle', 'Canis lupus familiaris');")
conn2b.commit()

print("\n=== Animal table after deleting ID 3 and inserting a new row ===")
run(conn2b, "SELECT * FROM Animal;")
print()
print("Notice: the new row received ID 4, not 3.")
print("AUTOINCREMENT prevents reuse of the deleted value.")

print("\n=== sqlite_sequence after the fourth insert ===")
run(conn2b, "SELECT * FROM sqlite_sequence;")

print("\n=== sqlite_master — schema catalog ===")
run(conn2b, "SELECT name, sql FROM sqlite_master WHERE type = 'table';")


=== Animal table after three inserts ===
animal_id  common_name  species               
---------  -----------  ----------------------
1          Mango        Canis lupus familiaris
2          Shadow       Felis catus           
3          Biscuit      Oryctolagus cuniculus 

(3 rows)

=== sqlite_sequence ===
name    seq
------  ---
Animal  3  

(1 row)

=== Animal table after deleting ID 3 and inserting a new row ===
animal_id  common_name  species               
---------  -----------  ----------------------
1          Mango        Canis lupus familiaris
2          Shadow       Felis catus           
4          Noodle       Canis lupus familiaris

(3 rows)

Notice: the new row received ID 4, not 3.
AUTOINCREMENT prevents reuse of the deleted value.

=== sqlite_sequence after the fourth insert ===
name    seq
------  ---
Animal  4  

(1 row)

=== sqlite_master — schema catalog ===
name             sql                                                                                     

Once again, we can use a `VACUUM` statement to take a copy of the `main` RAM-only database from memory to secondary storage.

In [47]:
run(conn2b, "VACUUM main INTO 'pets.db'")
show_tables(conn2)

Database error occurred: output file already exists
name 
-----
Owner
Pet  

(2 rows)


After downloading 'pets.db' to your computer, you can view its contents with DBbrowser or any SQLite utility.

**Reflection — replace this text with your answers:**

> 1. The fourth animal received ID 4. It did not receive ID 3 because the table uses AUTOINCREMENT, which ensures that SQLite never reuses previously used IDs, even if a row is deleted.

> 2. The seq column represents the highest ID value that has been assigned for a table. SQLite updates it whenever a new row is inserted that increases the maximum ID in a table that uses AUTOINCREMENT.

> 3. The sql column contains the original SQL statement used to create the table (or other database object), such as the CREATE TABLE statement.

> 4. No, sqlite_sequence would not exist. It is only created when a table uses AUTOINCREMENT, because SQLite only needs to track the highest used ID when it must guarantee that IDs are never reused.


---

### Exercise 2.3 — Insert valid owner rows

Insert the following three owners into the `Owner` table.  Use a single `executemany`
call with a list of tuples.

| owner_id | full_name       | phone        |
|:---------|:----------------|:-------------|
| 1        | Maria Vasquez   | 718-555-0101 |
| 2        | James O'Brien   | 212-555-0188 |
| 3        | Priya Nair      | 347-555-0234 |


In [51]:
owners = [
    (1, "Maria Vasquez", "718-555-0101"),
    (2, "James O'Brien", "212-555-0188"),
    (3, "Priya Nair", "347-555-0234")
]

conn2.executemany(
    "INSERT INTO Owner (owner_id, full_name, phone) VALUES (?, ?, ?);",
    owners
)

conn2.commit()
run(conn2, "SELECT * FROM Owner;")


owner_id  full_name      phone       
--------  -------------  ------------
1         Maria Vasquez  718-555-0101
2         James O'Brien  212-555-0188
3         Priya Nair     347-555-0234

(3 rows)


---

### Exercise 2.4 — Attempt an invalid insert

Try to insert a pet whose `owner_id` does not exist in the `Owner` table.  The cell
below wraps the insert in a `try/except` block so the notebook can continue after the
error.

Complete the `INSERT` statement with `owner_id = 99` (which does not exist).  Read the
error message that SQLite raises and write a one-sentence explanation in the markdown
cell that follows.


In [53]:
invalid_insert = """
INSERT INTO Pet (pet_id, pet_name, species, owner_id)
VALUES (10, 'Buddy', 'Canis lupus familiaris', 99);
"""

try:
    conn2.execute(invalid_insert)
    conn2.commit()
    print("Insert succeeded (unexpected).")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError caught: {e}")


IntegrityError caught: FOREIGN KEY constraint failed


**Your explanation (replace this text):**

SQLite raised an IntegrityError because the owner_id value (99) does not exist in the Owner table, violating the foreign key constraint.

---

### Exercise 2.5 — Insert valid pet rows

Now insert the following pets using valid `owner_id` values.

| pet_id | pet_name | species | owner_id |
|:-------|:---------|:--------|:---------|
| 1      | Mango    | Dog     | 1        |
| 2      | Shadow   | Cat     | 1        |
| 3      | Biscuit  | Rabbit  | 2        |
| 4      | Noodle   | Dog     | 3        |


In [56]:
pets = [
    (1, "Mango", "Dog", 1),
    (2, "Shadow", "Cat", 1),
    (3, "Biscuit", "Rabbit", 2),
    (4, "Noodle", "Dog", 3)
]

conn2.executemany(
    "INSERT INTO Pet (pet_id, pet_name, species, owner_id) VALUES (?, ?, ?, ?);",
    pets
)

conn2.commit()
run(conn2, "SELECT * FROM Pet;")


pet_id  pet_name  species  owner_id
------  --------  -------  --------
1       Mango     Dog      1       
2       Shadow    Cat      1       
3       Biscuit   Rabbit   2       
4       Noodle    Dog      3       

(4 rows)


---

### Exercise 2.6 — Verify enforcement: delete a parent row

Try to delete owner 1 (Maria Vasquez), who has two pets in the `Pet` table.

Predict what will happen before you run the cell, then observe the result.


In [58]:
try:
    conn2.execute("DELETE FROM Owner WHERE owner_id = 1;")
    conn2.commit()
    print("Delete succeeded (unexpected).")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError caught: {e}")


IntegrityError caught: FOREIGN KEY constraint failed


**Reflection (replace this text):**

The delete failed because owner 1 is referenced by rows in the Pet table, and the foreign key constraint prevents deleting a parent row that still has dependent child rows. To allow the deletion, you must first delete the related rows in the Pet table, then delete the owner from the Owner table.


---
## Section 3 — Normalization — Recognition and Repair

### Background

**Normalization** is the process of organizing a database schema to reduce redundancy
and prevent update anomalies.  The three most commonly applied normal forms are:

| Normal Form | Key Rule |
|:------------|:---------|
| **1NF** | Every column holds a single, atomic value.  No repeating groups. |
| **2NF** | 1NF plus: every non-key column depends on the *whole* primary key (matters when the key is composite). |
| **3NF** | 2NF plus: no non-key column depends on another non-key column (no transitive dependencies). |

A **denormalized** table violates one or more of these rules.  Common symptoms include:

- The same value repeated across many rows (e.g., a customer's address on every order row).
- Columns that describe something other than the entity the table is meant to track.
- Difficulty updating one piece of information without touching many rows.

---

### Exercise 3.1 — Identify the normalization problem

The table below represents a course enrollment system stored as a single flat table.

```
EnrollmentFlat
-----------------------------------------------------------------------
student_id | student_name | student_email       | course_id | course_title         | credits | instructor
-----------|--------------|---------------------|-----------|----------------------|---------|----------
1          | Ana Torres   | atorres@bcc.cuny.edu | CIS3120   | Prog for Analytics   | 3       | Prof. Kim
1          | Ana Torres   | atorres@bcc.cuny.edu | CIS2200   | Business Data Comm   | 3       | Prof. Lau
2          | Ben Reyes    | breyes@bcc.cuny.edu  | CIS3120   | Prog for Analytics   | 3       | Prof. Kim
3          | Cleo Marsh   | cmarsh@bcc.cuny.edu  | CIS4800   | Database Management  | 3       | Prof. Osei
```

In the markdown cell below, answer the following questions:

1. What data is repeated across multiple rows?
2. What anomaly would occur if Prof. Kim's name changed?
3. What normal form does this table violate, and why?


**Your answers (replace this text):**

> 1. Student information (student_name, student_email) and course information (course_title, instructor, credits) are repeated across multiple rows for each enrollment.

> 2. If Prof. Kim’s name changed, it would need to be updated in multiple rows, and missing one would lead to inconsistent data.

> 3. This table violates 3NF because non-key attributes (like instructor and course_title) depend on other non-key attributes (course_id), creating transitive dependencies.


---

### Exercise 3.2 — Decompose the flat table into a normalized schema

The flat `EnrollmentFlat` table should be split into three tables:

- `Student (student_id, student_name, student_email)`
- `Course (course_id, course_title, credits, instructor)`
- `Enrollment (student_id, course_id)` — a junction table linking students to courses

The setup cell below creates `EnrollmentFlat` and populates it.  Your task is to:

1. Create the three normalized tables with appropriate primary and foreign keys.
2. Populate each table by selecting data from `EnrollmentFlat`.


In [64]:
# Setup — creates EnrollmentFlat. Run this cell first.
conn3 = sqlite3.connect(":memory:")
conn3.execute("PRAGMA foreign_keys = ON;")

conn3.execute("""
    CREATE TABLE EnrollmentFlat (
        student_id     INTEGER,
        student_name   TEXT,
        student_email  TEXT,
        course_id      TEXT,
        course_title   TEXT,
        credits        INTEGER,
        instructor     TEXT
    );
""")

conn3.executemany(
    "INSERT INTO EnrollmentFlat VALUES (?, ?, ?, ?, ?, ?, ?);",
    [
        (1, "Ana Torres",  "atorres@bcc.cuny.edu", "CIS3120", "Prog for Analytics",  3, "Prof. Kim"),
        (1, "Ana Torres",  "atorres@bcc.cuny.edu", "CIS2200", "Business Data Comm",  3, "Prof. Lau"),
        (2, "Ben Reyes",   "breyes@bcc.cuny.edu",  "CIS3120", "Prog for Analytics",  3, "Prof. Kim"),
        (3, "Cleo Marsh",  "cmarsh@bcc.cuny.edu",  "CIS4800", "Database Management", 3, "Prof. Osei"),
    ]
)
conn3.commit()
print("EnrollmentFlat loaded.")
run(conn3, "SELECT * FROM EnrollmentFlat;")


EnrollmentFlat loaded.
student_id  student_name  student_email         course_id  course_title         credits  instructor
----------  ------------  --------------------  ---------  -------------------  -------  ----------
1           Ana Torres    atorres@bcc.cuny.edu  CIS3120    Prog for Analytics   3        Prof. Kim 
1           Ana Torres    atorres@bcc.cuny.edu  CIS2200    Business Data Comm   3        Prof. Lau 
2           Ben Reyes     breyes@bcc.cuny.edu   CIS3120    Prog for Analytics   3        Prof. Kim 
3           Cleo Marsh    cmarsh@bcc.cuny.edu   CIS4800    Database Management  3        Prof. Osei

(4 rows)


In [65]:
# TODO: Create the Student table
#       Columns: student_id (INTEGER PRIMARY KEY), student_name (TEXT NOT NULL),
#                student_email (TEXT)
conn3.execute("""
CREATE TABLE Student (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    student_email TEXT
);
""")

# TODO: Create the Course table
#       Columns: course_id (TEXT PRIMARY KEY), course_title (TEXT NOT NULL),
#                credits (INTEGER), instructor (TEXT)
conn3.execute("""
CREATE TABLE Course (
    course_id TEXT PRIMARY KEY,
    course_title TEXT NOT NULL,
    credits INTEGER,
    instructor TEXT
);
""")

# TODO: Create the Enrollment table
#       Columns: student_id (INTEGER), course_id (TEXT)
#       Primary key: composite (student_id, course_id)
#       Foreign keys: student_id -> Student(student_id)
#                     course_id  -> Course(course_id)
conn3.execute("""
CREATE TABLE Enrollment (
    student_id INTEGER,
    course_id TEXT,
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES Student(student_id),
    FOREIGN KEY (course_id) REFERENCES Course(course_id)
);
""")

conn3.commit()
show_tables(conn3)


name          
--------------
Course        
Enrollment    
EnrollmentFlat
Student       

(4 rows)




---



## SIDEBAR: `IS NULL`  
  
In SQLite, `NULL` means “unknown” or “missing,” not zero and not an empty string.  

That is why a test like `WHERE email = NULL` fails to find rows, even if some emails are missing.  

The correct form is `WHERE email IS NULL`, which returns rows where the column truly has no value.  

The opposite check is `IS NOT NULL`, which finds rows where the column does contain a value.  

In [68]:
conn4 = sqlite3.connect(":memory:")
cur = conn4.cursor()

cur.execute("""
CREATE TABLE students (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT
)
""")

cur.executemany(
    "INSERT INTO students (name, email) VALUES (?, ?)",
    [
        ("Alice", "alice@example.com"),
        ("Bob", None),
        ("Carol", "carol@example.com"),
        ("Dave", None),
    ],
)

print("Rows where email IS NULL:")
for row in cur.execute("SELECT id, name, email FROM students WHERE email IS NULL"):
    print(row)

print("\nRows where email IS NOT NULL:")
for row in cur.execute("SELECT id, name, email FROM students WHERE email IS NOT NULL"):
    print(row)

print('')

close_connection(conn4)

Rows where email IS NULL:
(2, 'Bob', None)
(4, 'Dave', None)

Rows where email IS NOT NULL:
(1, 'Alice', 'alice@example.com')
(3, 'Carol', 'carol@example.com')

Connection closed.


---
## Section 4 — Multi-Table Querying with Real Relationships

### Background

Once data is spread across related tables, answering most business questions requires
combining rows from two or more tables.

- An **INNER JOIN** (or simply `JOIN`) returns only rows where the join condition is
  satisfied in *both* tables.
- A **LEFT JOIN** returns all rows from the left table and matching rows from the right
  table.  Where there is no match, the right-side columns appear as `NULL`.  This is
  useful for finding records with *no* matching child rows.

In this section you will query the library database introduced in Module 7 — expanded
here to include a `Fine` table.

---

### Exercise 4.1 — Setup: the expanded library database


In [71]:
conn4 = sqlite3.connect(":memory:")
conn4.execute("PRAGMA foreign_keys = ON;")

conn4.executescript("""
    CREATE TABLE Member (
        member_id    INTEGER PRIMARY KEY,
        full_name    TEXT    NOT NULL,
        join_date    TEXT    NOT NULL
    );

    CREATE TABLE Book (
        book_id      INTEGER PRIMARY KEY,
        title        TEXT    NOT NULL,
        author       TEXT    NOT NULL,
        genre        TEXT
    );

    CREATE TABLE Loan (
        loan_id      INTEGER PRIMARY KEY,
        member_id    INTEGER NOT NULL,
        book_id      INTEGER NOT NULL,
        loan_date    TEXT    NOT NULL,
        return_date  TEXT,
        FOREIGN KEY (member_id) REFERENCES Member(member_id),
        FOREIGN KEY (book_id)   REFERENCES Book(book_id)
    );

    CREATE TABLE Fine (
        fine_id      INTEGER PRIMARY KEY,
        loan_id      INTEGER NOT NULL UNIQUE,
        amount       REAL    NOT NULL,
        paid         INTEGER NOT NULL DEFAULT 0,
        FOREIGN KEY (loan_id) REFERENCES Loan(loan_id)
    );
""")

conn4.executemany("INSERT INTO Member VALUES (?, ?, ?);", [
    (1, "Amara Osei",     "2019-03-15"),
    (2, "Luis Pereira",   "2021-07-22"),
    (3, "Yuki Tanaka",    "2020-11-01"),
    (4, "Grace Nwosu",    "2018-06-10"),
])

conn4.executemany("INSERT INTO Book VALUES (?, ?, ?, ?);", [
    (1,  "Invisible Man",          "Ralph Ellison",    "Fiction"),
    (2,  "Freakonomics",           "Levitt & Dubner",  "Non-Fiction"),
    (3,  "The Alchemist",          "Paulo Coelho",     "Fiction"),
    (4,  "Thinking, Fast and Slow","Daniel Kahneman",  "Non-Fiction"),
    (5,  "Americanah",             "Chimamanda Adichie","Fiction"),
])

conn4.executemany("INSERT INTO Loan VALUES (?, ?, ?, ?, ?);", [
    (1, 1, 1, "2023-01-10", "2023-01-28"),
    (2, 1, 3, "2023-02-05", "2023-03-01"),
    (3, 2, 2, "2023-01-20", "2023-02-10"),
    (4, 3, 4, "2023-03-01", None),
    (5, 4, 5, "2022-12-01", "2023-01-15"),
    (6, 4, 1, "2023-04-10", None),
])

conn4.executemany("INSERT INTO Fine VALUES (?, ?, ?, ?);", [
    (1, 2, 3.50, 0),
    (2, 5, 7.00, 1),
])

conn4.commit()
print("Library database loaded.")
show_tables(conn4)


Library database loaded.
name  
------
Book  
Fine  
Loan  
Member

(4 rows)


Once again, we can use a `VACUUM` statement to take a copy of the `main` RAM-only database from memory to secondary storage.

In [73]:
run(conn4, "VACUUM main INTO 'library.db'")
show_tables(conn4)

Database error occurred: output file already exists
name  
------
Book  
Fine  
Loan  
Member

(4 rows)


---

### Exercise 4.2 — List all loans with member name and book title

Write a query that returns each loan with the borrowing member's name and the book
title.  Include: `loan_id`, `full_name`, `title`, `loan_date`, and `return_date`.
Order the results by `loan_date`.


In [75]:
query_4_2 = """
SELECT 
    l.loan_id,
    m.full_name,
    b.title,
    l.loan_date,
    l.return_date
FROM Loan l
JOIN Member m ON l.member_id = m.member_id
JOIN Book b ON l.book_id = b.book_id
ORDER BY l.loan_date;
"""

run(conn4, query_4_2)


loan_id  full_name     title                    loan_date   return_date
-------  ------------  -----------------------  ----------  -----------
5        Grace Nwosu   Americanah               2022-12-01  2023-01-15 
1        Amara Osei    Invisible Man            2023-01-10  2023-01-28 
3        Luis Pereira  Freakonomics             2023-01-20  2023-02-10 
2        Amara Osei    The Alchemist            2023-02-05  2023-03-01 
4        Yuki Tanaka   Thinking, Fast and Slow  2023-03-01  None       
6        Grace Nwosu   Invisible Man            2023-04-10  None       

(6 rows)


---

### Exercise 4.3 — Find members who have never borrowed a book

Use a `LEFT JOIN` to identify members who have no rows in the `Loan` table.

Hint: after a `LEFT JOIN`, a member with no loans will have `NULL` in any
`Loan` column.  Use `WHERE loan_id IS NULL` to filter for those rows.


In [77]:
query_4_3 = """
SELECT 
    m.member_id,
    m.full_name
FROM Member m
LEFT JOIN Loan l ON m.member_id = l.member_id
WHERE l.loan_id IS NULL;
"""

run(conn4, query_4_3)


Statement executed successfully. No rows returned.


---

### Exercise 4.4 — List all outstanding loans with any associated fine amount

Write a query that shows loans where `return_date IS NULL`.  For each such loan,
include the member name, book title, loan date, and the fine amount if one exists
(show `NULL` if no fine has been issued).

Hint: use an `INNER JOIN` to reach `Member` and `Book`, and a `LEFT JOIN` to
reach `Fine` (since not every outstanding loan has a fine).


In [79]:
query_4_4 = """
SELECT
    m.full_name,
    b.title,
    l.loan_date,
    f.amount
FROM Loan l
JOIN Member m ON l.member_id = m.member_id
JOIN Book b ON l.book_id = b.book_id
LEFT JOIN Fine f ON l.loan_id = f.loan_id
WHERE l.return_date IS NULL;
"""

run(conn4, query_4_4)


full_name    title                    loan_date   amount
-----------  -----------------------  ----------  ------
Yuki Tanaka  Thinking, Fast and Slow  2023-03-01  None  
Grace Nwosu  Invisible Man            2023-04-10  None  

(2 rows)


---

### Exercise 4.5 — Aggregate: total fines owed per member

Write a query that returns each member's name and the total unpaid fine amount
they owe.  Only include members who have at least one unpaid fine (`paid = 0`).

Use `JOIN`, `GROUP BY`, and the `SUM()` aggregate function.


In [81]:
query_4_5 = """
SELECT
    m.full_name,
    SUM(f.amount) AS total_unpaid_fines
FROM Loan l
JOIN Fine f ON l.loan_id = f.loan_id
JOIN Member m ON l.member_id = m.member_id
WHERE f.paid = 0
GROUP BY m.full_name;
"""

run(conn4, query_4_5)


full_name   total_unpaid_fines
----------  ------------------
Amara Osei  3.5               

(1 row)


---
## Section 5 — Data Integrity Scenarios

### Background

Foreign key constraints protect data integrity but also impose an ordering requirement
on operations:

- You must **insert parent rows before child rows**.
- You must **delete child rows before parent rows** (or use `ON DELETE CASCADE`).
- Updating a primary key value that is referenced by a foreign key requires similar care.

This section gives you practice recognizing and resolving the most common integrity
errors.

---

### Exercise 5.1 — Correct deletion order

The cell below attempts to delete book 1 (*Invisible Man*) directly from the `Book`
table.  This will fail because `Loan` rows reference it.

1. Run the cell and read the error.
2. In the second cell, write a corrected deletion sequence that removes all loans for
   book 1 first, then deletes the book itself.

Note: you are working on `conn4` from Section 4, which already has data loaded.


In [84]:
try:
    conn4.execute("DELETE FROM Book WHERE book_id = 1;")
    conn4.commit()
    print("Delete succeeded (unexpected).")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e}")
    conn4.rollback()


IntegrityError: FOREIGN KEY constraint failed


In [85]:
# TODO: First, delete any Fine rows linked to loans for book_id = 1
#       Hint: Fine references Loan, so fines must be deleted before loans
conn4.execute("""
DELETE FROM Fine
WHERE loan_id IN (
    SELECT loan_id
    FROM Loan
    WHERE book_id = 1
);
""")

# TODO: Next, delete the Loan rows for book_id = 1
conn4.execute("""
DELETE FROM Loan
WHERE book_id = 1;
""")

# TODO: Finally, delete the Book row
conn4.execute("""
DELETE FROM Book
WHERE book_id = 1;
""")


conn4.commit()
print("Deletion sequence completed.")
run(conn4, "SELECT * FROM Book;")


Deletion sequence completed.
book_id  title                    author              genre      
-------  -----------------------  ------------------  -----------
2        Freakonomics             Levitt & Dubner     Non-Fiction
3        The Alchemist            Paulo Coelho        Fiction    
4        Thinking, Fast and Slow  Daniel Kahneman     Non-Fiction
5        Americanah               Chimamanda Adichie  Fiction    

(4 rows)


---

### Exercise 5.2 — PRAGMA foreign_keys OFF vs ON

This exercise demonstrates why the PRAGMA matters.

The cell below connects to a *new* in-memory database **without** enabling foreign keys.
It creates the same `Owner` / `Pet` schema from Section 2 but inserts a pet with a
non-existent `owner_id`.  Run it and observe that no error is raised.

Then modify the cell to add `conn5.execute("PRAGMA foreign_keys = ON;")` immediately
after the connection is opened, re-run, and observe the difference.


In [87]:
conn5 = sqlite3.connect(":memory:")
conn5.execute("PRAGMA foreign_keys = ON;")

conn5.execute("""
    CREATE TABLE Owner (
        owner_id  INTEGER PRIMARY KEY,
        full_name TEXT NOT NULL
    );
""")
conn5.execute("""
    CREATE TABLE Pet (
        pet_id    INTEGER PRIMARY KEY,
        pet_name  TEXT    NOT NULL,
        owner_id  INTEGER NOT NULL,
        FOREIGN KEY (owner_id) REFERENCES Owner(owner_id)
    );
""")
conn5.commit()

try:
    conn5.execute("INSERT INTO Pet VALUES (1, 'Ghost', 999);")
    conn5.commit()
    print("Insert succeeded — foreign key was NOT enforced.")
except sqlite3.IntegrityError as e:
    print(f"IntegrityError: {e} — foreign key WAS enforced.")


IntegrityError: FOREIGN KEY constraint failed — foreign key WAS enforced.


**Reflection (replace this text):**

> SQLite defaults foreign key enforcement to OFF for backward compatibility with older databases that may not enforce relational constraints. This means Python programs using sqlite3 must explicitly enable PRAGMA foreign_keys = ON, otherwise invalid foreign key values can be inserted without errors.


---
## Section 6 — Schema Design Challenge

### The scenario

A community gym wants to track its **members**, the **fitness classes** it offers, and
member **enrollments** in those classes.  The following rules apply:

- Each member has an ID, a full name, a membership start date, and a membership type
  (`"Basic"` or `"Premium"`).
- Each fitness class has an ID, a class name, an instructor name, a day of the week,
  and a maximum capacity (integer).
- A member can enroll in many classes; a class can have many members (many-to-many).
- An enrollment records which member joined which class and the date they enrolled.
- A member may not enroll in the same class more than once.


---

### Exercise 6.1 — Design the schema

In the markdown cell below, list the three tables you will create, their columns, and
identify the primary key and any foreign keys for each table.  Use plain text — no SQL
yet.


**Your schema design (replace this text):**

> **Member table**
> Columns: member_id, full_name, email, phone
Primary key: member_id


> **FitnessClass table**
> Columns: class_id, class_name, instructor, schedule_time
Primary key: class_id

> **Enrollment table**
> Columns: member_id, class_id, enrollment_date
Primary key: composite key (member_id, class_id)
Foreign keys:
member_id references Member(member_id)
class_id references FitnessClass(class_id)


---

### Exercise 6.2 — Implement the schema

Write and execute the three `CREATE TABLE` statements.  Remember to:

- Enable `PRAGMA foreign_keys = ON`.
- Use a composite primary key on the `Enrollment` table to prevent duplicate enrollments.
- Declare `FOREIGN KEY` constraints on `Enrollment`.


In [93]:
conn6 = sqlite3.connect(":memory:")
conn6.execute("PRAGMA foreign_keys = ON;")

# TODO: CREATE TABLE Member
conn6.execute("""
CREATE TABLE Member (
    member_id INTEGER PRIMARY KEY,
    full_name TEXT NOT NULL,
    email TEXT,
    phone TEXT
);
""")

# TODO: CREATE TABLE FitnessClass
conn6.execute("""
CREATE TABLE FitnessClass (
    class_id INTEGER PRIMARY KEY,
    class_name TEXT NOT NULL,
    instructor TEXT,
    schedule_time TEXT
);
""")


# TODO: CREATE TABLE Enrollment
conn6.execute("""
CREATE TABLE Enrollment (
    member_id INTEGER,
    class_id INTEGER,
    enrollment_date TEXT,
    PRIMARY KEY (member_id, class_id),
    FOREIGN KEY (member_id) REFERENCES Member(member_id),
    FOREIGN KEY (class_id) REFERENCES FitnessClass(class_id)
);
""")

conn6.commit()
show_tables(conn6)


name        
------------
Enrollment  
FitnessClass
Member      

(3 rows)


---

### Exercise 6.3 — Populate the database

Insert at least three members, three fitness classes, and five enrollments of your
choice.  At least one member should be enrolled in more than one class.


In [95]:
# TODO: insert at least 3 members
conn6.executemany(
    "INSERT INTO Member VALUES (?, ?, ?, ?);",
    [
        (1, "Jordan Miles", "jordan.miles@email.com", "646-555-0142"),
        (2, "Nia Bennett", "nia.bennett@email.com", "718-555-0268"),
        (3, "Omar Castillo", "omar.castillo@email.com", "917-555-0315")
    ]
)

# TODO: insert at least 3 fitness classes
conn6.executemany(
    "INSERT INTO FitnessClass VALUES (?, ?, ?, ?);",
    [
        (1, "Pilates Core", "Dana Brooks", "Tuesday 6:00 PM"),
        (2, "Strength Circuit", "Marcus Reed", "Thursday 7:00 PM"),
        (3, "Zumba Blast", "Elena Cruz", "Saturday 10:00 AM")
    ]
)

# TODO: insert at least 5 enrollments
conn6.executemany(
    "INSERT INTO Enrollment VALUES (?, ?, ?);",
    [
        (1, 1, "2026-04-20"),
        (1, 2, "2026-04-21"),
        (2, 1, "2026-04-22"),
        (2, 3, "2026-04-23"),
        (3, 2, "2026-04-24")
    ]
)

conn6.commit()
print("=== Member ===")
run(conn6, "SELECT * FROM Member;")
print("\n=== FitnessClass ===")
run(conn6, "SELECT * FROM FitnessClass;")
print("\n=== Enrollment ===")
run(conn6, "SELECT * FROM Enrollment;")


=== Member ===
member_id  full_name      email                    phone       
---------  -------------  -----------------------  ------------
1          Jordan Miles   jordan.miles@email.com   646-555-0142
2          Nia Bennett    nia.bennett@email.com    718-555-0268
3          Omar Castillo  omar.castillo@email.com  917-555-0315

(3 rows)

=== FitnessClass ===
class_id  class_name        instructor   schedule_time    
--------  ----------------  -----------  -----------------
1         Pilates Core      Dana Brooks  Tuesday 6:00 PM  
2         Strength Circuit  Marcus Reed  Thursday 7:00 PM 
3         Zumba Blast       Elena Cruz   Saturday 10:00 AM

(3 rows)

=== Enrollment ===
member_id  class_id  enrollment_date
---------  --------  ---------------
1          1         2026-04-20     
1          2         2026-04-21     
2          1         2026-04-22     
2          3         2026-04-23     
3          2         2026-04-24     

(5 rows)


---

### Exercise 6.4 — Query your database

Write queries to answer the following questions about your gym data.

**Query A**: List each member's full name alongside the names of the classes they
are enrolled in.


In [143]:
query_6_a = """
SELECT
    m.full_name,
    f.class_name
FROM Member m
JOIN Enrollment e
    ON m.member_id = e.member_id
JOIN FitnessClass f
    ON e.class_id = f.class_id;
"""

run(conn6, query_6_a)


full_name      class_name      
-------------  ----------------
Jordan Miles   Pilates Core    
Jordan Miles   Strength Circuit
Nia Bennett    Pilates Core    
Nia Bennett    Zumba Blast     
Omar Castillo  Strength Circuit

(5 rows)


**Query B**: Find any fitness classes that currently have *no* enrollments.


In [147]:
query_6_b = """
SELECT
    f.class_name
FROM FitnessClass f
LEFT JOIN Enrollment e
    ON f.class_id = e.class_id
WHERE e.member_id IS NULL;
"""

run(conn6, query_6_b)


Statement executed successfully. No rows returned.


**Query C**: For each class, show the class name and the count of enrolled members.
Order the results from most to least enrolled.


In [149]:
query_6_c = """
SELECT
    f.class_name,
    COUNT(e.member_id) AS enrolled_count
FROM FitnessClass f
LEFT JOIN Enrollment e
    ON f.class_id = e.class_id
GROUP BY f.class_id, f.class_name
ORDER BY enrolled_count DESC;
"""

run(conn6, query_6_c)


class_name        enrolled_count
----------------  --------------
Pilates Core      2             
Strength Circuit  2             
Zumba Blast       1             

(3 rows)


---
## Notebook Complete

You have worked through:

- Enabling and testing foreign key enforcement in SQLite
- Inserting and deleting rows while respecting referential integrity
- Identifying normalization problems in a flat table and decomposing it into a
  properly related schema
- Writing `INNER JOIN` and `LEFT JOIN` queries that traverse foreign key relationships
- Designing and implementing a normalized three-table schema from a written specification

**Submission**: Push this completed notebook to your GitHub repository and submit the
link on Brightspace.
